In [ ]:
import signal, time as _t
_T0 = _t.time()
def _cap(*a): raise TimeoutError("25 min cap hit after %.0fs" % (_t.time() - _T0))
signal.signal(signal.SIGALRM, _cap); signal.alarm(25 * 60)

# ===== UNTESTED: RevealLayer on Kaggle 2xT4, GGUF Q4_K_S FLUX.1-dev, fp16, 512px =====
import subprocess, sys, os, json
def sh(c): print(c); subprocess.run(c, shell=True, check=True)
sh("pip -q install transformers==4.57.1 gguf>=0.10 peft==0.15.2 mmengine==0.10.7 einops sentencepiece protobuf")
RL="/kaggle/working/RevealLayer"; COMMIT="da68ff58889a1799a1e75efd23e72bccebbf7589"
if not os.path.exists(RL):
    sh(f"git clone https://github.com/360CVGroup/RevealLayer {RL} && cd {RL} && git checkout {COMMIT}")

# ---- source patches (python string replace) ----
def patch(path, old, new, count=-1):
    p=f"{RL}/{path}"; s=open(p,encoding="utf-8").read()
    assert old in s, f"patch anchor not found in {path}: {old[:60]}"
    open(p,"w",encoding="utf-8").write(s.replace(old,new,count)); print("patched",path)
patch("infer.py", "torch.bfloat16", "torch.float16")
patch("infer.py", "resolution=1024,", "resolution=512,")
patch("infer.py", "guidance_scale=args.cfg,", "guidance_scale=args.cfg, true_gs=getattr(args,'true_gs',3.5),")
patch("models/custom_model_mmdit.py", "-1e9, dtype=dtype", "torch.finfo(dtype).min, dtype=dtype")
patch("models/custom_pipeline.py", "neg_noise_pred = self.transformer(",
      "neg_noise_pred = noise_pred if true_gs == 1.0 else self.transformer(")


os.makedirs('/kaggle/working/vit32', exist_ok=True)
sh('wget -q -nc -O /kaggle/working/vit32/vit_b_32-d86f8d99.pth https://download.pytorch.org/models/vit_b_32-d86f8d99.pth')
import glob as _g, re as _re
sh('mkdir -p /home/jovyan/boomcheng-work-shcdt/zhaoshihao/pretrained_model/vit32 && cp /kaggle/working/vit32/vit_b_32-d86f8d99.pth /home/jovyan/boomcheng-work-shcdt/zhaoshihao/pretrained_model/vit32/')
for _f in _g.glob(RL + '/**/*.py', recursive=True):
    _s = open(_f, encoding='utf-8', errors='ignore').read()
    if 'vit32' in _s and '/home/jovyan' in _s:
        _s = _re.sub(r"/home/jovyan/[^'\"]*?vit32", "/kaggle/working/vit32", _s)
        open(_f, 'w', encoding='utf-8').write(_s); print('vit path patched in', _f)

patch("models/custom_model_mmdit.py", "torch.full((t_total, t_total), -1e9, dtype=dtype", "torch.full((t_total, t_total), -1e4, dtype=dtype")
patch("models/custom_model_mmdit.py", "hidden_states_part2 = hidden_states[:, l0_len:, :] + delta",
      "hidden_states_part2 = (hidden_states[:, l0_len:, :].float() + torch.nan_to_num(delta.float(), nan=0.0, posinf=6e4, neginf=-6e4)).clamp(-6e4, 6e4).to(hidden_states.dtype)")
patch("models/custom_pipeline.py", "image_segs = [self.vae.decode(latents_seg, return_dict=False)[0] for latents_seg in latents_segs]",
      "image_segs = [self.vae.float().decode(latents_seg.float(), return_dict=False)[0] for latents_seg in latents_segs]")
# ---- use bundled diffusers 0.34 (has GGUF + attention_mask-aware Flux blocks) ----
sys.path[:0] = [RL, f"{RL}/diffusers/src"]
for m in [k for k in sys.modules if k.startswith("diffusers")]: del sys.modules[m]
import torch, torch.nn as nn
from huggingface_hub import hf_hub_download, snapshot_download
import diffusers; print("diffusers", diffusers.__version__, diffusers.__file__)
from diffusers import FluxTransformer2DModel, AutoencoderKL, GGUFQuantizationConfig, FlowMatchEulerDiscreteScheduler
from safetensors.torch import load_file

# ---- downloads ----
gguf = hf_hub_download("city96/FLUX.1-dev-gguf", "flux1-dev-Q4_K_S.gguf")
CK = snapshot_download("qihoo360/RevealLayer", allow_patterns=["*.pt","*.safetensors","xvae/*"])
BASE = snapshot_download("ostris/Flex.1-alpha",  # ungated; FLUX VAE + CLIP-L + T5-XXL
    allow_patterns=["model_index.json","text_encoder/*","text_encoder_2/*","tokenizer/*","tokenizer_2/*","vae/*","scheduler/*"])
CFG = "/kaggle/working/flux_dev_cfg"; os.makedirs(f"{CFG}/transformer", exist_ok=True)
json.dump({"_class_name":"FluxTransformer2DModel","attention_head_dim":128,"axes_dims_rope":[16,56,56],
           "guidance_embeds":True,"in_channels":64,"joint_attention_dim":4096,"num_attention_heads":24,
           "num_layers":19,"num_single_layers":38,"patch_size":1,"pooled_projection_dim":768},
          open(f"{CFG}/transformer/config.json","w"))
dev_sched = FlowMatchEulerDiscreteScheduler(num_train_timesteps=1000, shift=3.0, use_dynamic_shifting=True,
            base_shift=0.5, max_shift=1.15, base_image_seq_len=256, max_image_seq_len=4096)

D0, D1, DT = torch.device("cuda:0"), torch.device("cuda:1"), torch.float16
from models.custom_model_mmdit import CustomFluxTransformer2DModel
from models.adapter import HybridRefiner
from models.custom_pipeline import CustomFluxPipelineCfg
from models.custom_model_xvae import AutoencoderKLTransformerTraining as CustomVAE
import infer  # helpers + (patched) test_one_sample

# ---- R1: GGUF transformer, re-classed into the custom model ----
t = FluxTransformer2DModel.from_single_file(gguf, config=CFG, subfolder="transformer",
        quantization_config=GGUFQuantizationConfig(compute_dtype=DT), torch_dtype=DT)
t.__class__ = CustomFluxTransformer2DModel
t.register_to_config(max_layer_num=12); t.max_layer_num = 12
t.layer_pe = nn.Parameter(torch.load(f"{CK}/layer_pe.pt", map_location="cpu").to(DT).view(1,12,1,1,3072))
t.refiner = HybridRefiner(feature_dim=3072, pixel_dim=16, time_emb_dim=3072, attn_hidden=1024, num_layers=1, patch_size=2)
print("refiner:", t.refiner.load_state_dict(torch.load(f"{CK}/Refiner.pt", map_location="cpu"), strict=False))
t.refiner.to(DT); t.to(D0).eval()


# ---- fp16_mixed (layerproto/fp16_mixed.py, inlined) ----
exec(compile('"""Run the Qwen-Image(-Layered) transformer on GPUs without bf16 (Kaggle T4, P100, RTX 20xx) without NaNs.\n\nQwen-Image is trained for bf16; its hidden values exceed the fp16 limit (65504), so a plain fp16 run gives all-NaN\nlatents (ComfyUI also lists Qwen-Image as bf16/fp32 only). This keeps the fast fp16 tensor-core matmuls but holds\neverything else in fp32:\n\n  * every Linear: input scaled down to absmax 32, fp16 matmul, result scaled back in fp32\n    (falls back to a fp32 matmul for that call if the fp16 result is still non-finite)\n  * attention: q/k in fp16 (RMS-normalised, small), v scaled the same way (attention is linear in v)\n  * residual stream, norms, gates, embeddings: fp32\n  * the model\'s output is returned in fp16, so the diffusers pipeline is unchanged\n\nMeasured on Kaggle 2x T4 (Q4_K_S GGUF, 640 px, 4 layers): no NaN, 42 s/step, peak 8.8 / 8.4 GB.\n\n    from layerproto.fp16_mixed import apply_fp16_mixed\n    apply_fp16_mixed(transformer)        # once, after loading; idempotent\n"""\nimport types\n\nimport torch\nimport torch.nn.functional as F\n\nLINEAR_LIMIT = 32.0  # absmax the fp16 matmul input is scaled to (1024 was still too high: NaNs)\nATTN_LIMIT = 32.0\n\n\ndef _dequant(w):\n    if type(w).__name__ == "GGUFParameter":\n        from diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\n        return dequantize_gguf_tensor(w)\n    return w\n\n\ndef _mixed_linear(self, x):\n    x = x.float()\n    s = (x.abs().amax() / LINEAR_LIMIT).clamp_(min=1.0)\n    y = self._fp16_orig_forward((x / s).to(torch.float16)).float()   # = W x/s + b\n    b = getattr(self, "bias", None)\n    if not torch.isfinite(y).all():                                  # rare: this call in fp32\n        y = F.linear(x, _dequant(self.weight).float(), None if b is None else b.float())\n    elif b is None:\n        y = y * s\n    else:\n        b = b.float()\n        y = (y - b) * s + b\n    lora = getattr(self, "_fp16_lora", None)\n    if lora is not None:                                             # low-rank delta, same input scaling\n        A, B, k = lora\n        if A.device != x.device:                                     # module was moved (e.g. split over GPUs)\n            A, B = A.to(x.device), B.to(x.device)\n            self._fp16_lora = (A, B, k)\n        xs = (x / s).to(torch.float16)\n        y = y + F.linear(F.linear(xs, A), B).float() * (s * k)\n    return y\n\n\ndef _lora_pairs(sd):\n    """{module name: (A [r, in], B [out, r], alpha or None)} for common LoRA key layouts (PEFT / kohya / diffusers)."""\n    out = {}\n    for key in sd:\n        for a_suf, b_suf in ((".lora_A.weight", ".lora_B.weight"), (".lora_down.weight", ".lora_up.weight"),\n                             (".lora_A.default.weight", ".lora_B.default.weight")):\n            if key.endswith(a_suf):\n                base = key[: -len(a_suf)]\n                bkey = base + b_suf\n                if bkey in sd:\n                    alpha = sd.get(base + ".alpha")\n                    name = base\n                    for pre in ("base_model.model.", "diffusion_model.", "transformer.", "model."):\n                        if name.startswith(pre):\n                            name = name[len(pre):]\n                    out[name] = (sd[key], sd[bkey], None if alpha is None else float(alpha))\n    return out\n\n\ndef add_lora(model, path_or_state_dict, scale: float = 1.0):\n    """Attach a LoRA to an fp16_mixed-patched model (works with GGUF-quantised linears, nothing is merged).\n    Returns (matched modules, LoRA modules not found)."""\n    sd = path_or_state_dict\n    if isinstance(sd, str):\n        from safetensors.torch import load_file\n        sd = load_file(sd)\n    pairs = _lora_pairs(sd)\n    mods = dict(model.named_modules())\n    hit, miss = 0, []\n    for name, (A, B, alpha) in pairs.items():\n        m = mods.get(name)\n        if m is None or not hasattr(m, "_fp16_orig_forward"):\n            miss.append(name)\n            continue\n        dev = next((p.device for p in m.parameters()), torch.device("cpu"))\n        r = A.shape[0]\n        k = scale * ((alpha / r) if alpha else 1.0)\n        m._fp16_lora = (A.to(dev, torch.float16), B.to(dev, torch.float16), k)\n        hit += 1\n    return hit, miss\n\n\ndef remove_lora(model):\n    for _, m in model.named_modules():\n        if hasattr(m, "_fp16_lora"):\n            del m._fp16_lora\n\n\ndef _patch_attention():\n    import diffusers.models.transformers.transformer_qwenimage as tq\n    if getattr(tq, "_fp16_orig_attn", None) is not None:\n        return\n    orig = tq.dispatch_attention_fn\n\n    def attn_fp16(q, k, v, attn_mask=None, **kw):\n        s = (v.float().abs().amax() / ATTN_LIMIT).clamp_(min=1.0)\n        if attn_mask is not None and attn_mask.is_floating_point():\n            attn_mask = attn_mask.to(torch.float16)\n        o = orig(q.to(torch.float16), k.to(torch.float16), (v.float() / s).to(torch.float16), attn_mask=attn_mask, **kw)\n        return o.float() * s\n\n    tq._fp16_orig_attn = orig\n    tq.dispatch_attention_fn = attn_fp16\n\n\ndef apply_fp16_mixed(model):\n    """Patch a loaded QwenImageTransformer2DModel in place. Returns (linears patched, params moved to fp32)."""\n    _patch_attention()\n    n_lin = n_par = 0\n    for _, m in model.named_modules():\n        if "Linear" in type(m).__name__:\n            if not hasattr(m, "_fp16_orig_forward"):\n                m._fp16_orig_forward = m.forward\n            m.forward = types.MethodType(_mixed_linear, m)\n            n_lin += 1\n        else:\n            for _, p in m.named_parameters(recurse=False):\n                if p.dtype == torch.float16:\n                    p.data = p.data.float()\n                    n_par += 1\n    if not hasattr(model, "_fp16_orig_model_forward"):\n        model._fp16_orig_model_forward = model.forward\n\n        def fwd(*a, **k):\n            o = model._fp16_orig_model_forward(*a, **k)\n            if isinstance(o, tuple):\n                return (o[0].to(torch.float16),) + tuple(o[1:])\n            if hasattr(o, "sample"):\n                o.sample = o.sample.to(torch.float16)\n            return o\n\n        model.forward = fwd\n    return n_lin, n_par\n', 'fp16_mixed', 'exec'))
apply_fp16_mixed(t); print('fp16_mixed applied', flush=True)
# ---- R7: fp16 overflow guard ----
_hit = {"n":0}
def guard(mod, inp, out):
    def fix(o):
        if torch.is_tensor(o) and o.dtype == torch.float16 and not torch.isfinite(o).all():
            _hit["n"] += 1
            if _hit["n"] == 1: print("[guard] non-finite fp16 activations in", type(mod).__name__)
            return torch.nan_to_num(o, nan=0.0, posinf=6e4, neginf=-6e4)
        return o
    return tuple(fix(o) for o in out) if isinstance(out, tuple) else fix(out)
for b in list(t.transformer_blocks) + list(t.single_transformer_blocks) + [t.refiner]:
    b.register_forward_hook(guard)

# ---- R2/R3: pipeline, components placed by hand ----
vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=DT)
pipe = CustomFluxPipelineCfg.from_pretrained(BASE, transformer=t, vae=vae, scheduler=dev_sched, torch_dtype=DT)
pipe.vae.to(D0); pipe.text_encoder.to(D0); pipe.text_encoder_2.to(D1)
CustomFluxPipelineCfg._execution_device = property(lambda self: D0)
_orig_enc = pipe.encode_prompt
def _enc(*a, device=None, **kw):
    # run CLIP on cuda:0 inputs and T5 on cuda:1 inputs, then move outputs to cuda:0
    pipe.text_encoder.to(D1); out = _orig_enc(*a, device=D1, **kw); pipe.text_encoder.to(D0)
    return tuple(o.to(D0) if torch.is_tensor(o) else o for o in out)
pipe.encode_prompt = _enc

lora = {k.replace("transformer.module.", "transformer."): v for k, v in load_file(f"{CK}/pytorch_lora_weights.safetensors").items()}
pipe.load_lora_weights(lora, adapter_name="layer")
for n, p in pipe.transformer.named_parameters():
    if "lora_" in n: p.data = p.data.to(D0, DT)

transp_vae = CustomVAE()
print("xvae:", transp_vae.load_state_dict(torch.load(f"{CK}/xvae/transparent_decoder_ckpt.pth", map_location="cpu"), strict=False))
transp_vae.to(D0).eval()


# ---- our scenes (replaces the demo example) ----
import base64, io, time
from types import SimpleNamespace
from PIL import Image
SCENES = {"forest": ["...", [[0, 0, 260, 420], [300, 0, 512, 420], [100, 320, 340, 460]]], "sketch_skeleton": ["...", [[110, 240, 280, 350], [250, 100, 400, 370], [170, 270, 340, 420]]]}
for name, (b, boxes) in list(SCENES.items())[:1]:
    p = f"/kaggle/working/{name}.png"; Image.open(io.BytesIO(base64.b64decode(b))).save(p)
    sample = {"index": name, "layout": boxes, "full_image_path": p}
    for true_gs, steps in ((1.0, 12),):
        print(f"start {name} at {time.time()-_T0:.0f}s since cell start", flush=True)
        args = SimpleNamespace(seed=43, cfg=1.0, steps=steps, true_gs=true_gs, output_dir=f"/kaggle/working/reveal_out/gs{true_gs}_st{steps}")
        torch.cuda.reset_peak_memory_stats(0); torch.cuda.reset_peak_memory_stats(1); t0 = time.time()
        try:
            with torch.inference_mode():
                infer.test_one_sample(args, sample, pipe, transp_vae, D0, shuffle=False)
            print(f"{name} true_gs={true_gs} steps={steps}: {time.time()-t0:.0f}s; guard hits={_hit['n']}; peak GiB",
                  [round(torch.cuda.max_memory_allocated(i)/2**30, 1) for i in range(2)], flush=True)
        except BaseException as e:
            import traceback; traceback.print_exc(); print("FAILED", name, true_gs, steps, flush=True)
sh("find /kaggle/working/reveal_out -name '*.png' | head -50")